In [42]:
import sys
sys.path.append('/Users/eitanturok/good-vibrations/src3')

In [43]:
from pathlib import Path

In [44]:
BOX = 'cardboard'
SPEAKER = 2  # only keep samples recorded with this speaker

BASE_SAMPLE_DIR = Path(rf'D:\eturok\experiment-20\data\{BOX}\samples')

In [45]:
import json
import numpy as np
import pandas as pd
from itertools import combinations

In [46]:
def load_metadata(sample_dir):
    """Merge the single-key dicts in metadata.jsonl into one dict."""
    meta = {}
    with open(sample_dir / 'metadata.jsonl') as f:
        for line in f:
            line = line.strip()
            if line:
                meta.update(json.loads(line))
    return meta


rows = []
for sample_dir in sorted(BASE_SAMPLE_DIR.iterdir()):
    if not sample_dir.is_dir():
        continue
    meta = load_metadata(sample_dir)
    rows.append({
        'sample_id': meta['sample_id'],
        'speaker': meta['speaker'],
        'com': np.array(meta['com'], dtype=float),
        'fft_path': sample_dir / 'inputs' / '03_fft_shifts.npz',
        'mask_path': sample_dir / 'outputs' / '02_segment_mask.png',
    })

df = pd.DataFrame(rows)
df = df[df['speaker'] == SPEAKER]
# Sort by center of mass: first the x coordinate (com[0]), then y (com[1]).
order = np.lexsort((df['com'].apply(lambda c: c[1]).to_numpy(),
                    df['com'].apply(lambda c: c[0]).to_numpy()))
df = df.iloc[order].reset_index(drop=True)

# Per-sample colors keyed to each sample's position in df, shared by the FFT and
# segmentation plots so the same sample is the same color in both.
from plotly.colors import sample_colorscale
SAMPLE_COLORS = sample_colorscale('Viridis', list(np.linspace(0, 1, len(df))))
SAMPLE_COLOR_IDX = {sid: k for k, sid in enumerate(df['sample_id'])}
df

,sample_id,speaker,com,fft_path,mask_path
0,000073,2,"[-1.0, -1.0]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
1,000065,2,"[72.5762987012987, 134.17694805194805]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
2,000017,2,"[73.36605657237936, 182.27787021630616]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
3,000041,2,"[74.52380952380952, 157.4055829228243]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
4,000057,2,"[93.28410914927768, 132.93258426966293]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
5,000009,2,"[94.10726072607261, 181.04620462046205]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
6,000033,2,"[94.29256198347107, 156.5586776859504]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
7,000001,2,"[113.23205342237061, 178.35559265442404]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
8,000025,2,"[113.29883138564274, 154.2220367278798]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...
9,000049,2,"[114.24592833876221, 131.77524429967426]",D:\eturok\experiment-20\data\cardboard\samples...,D:\eturok\experiment-20\data\cardboard\samples...


In [47]:
import cv2
import numpy as np
import plotly.graph_objects as go
from PIL import Image

# Load every sample's segmentation mask, overlay their outlines in distinct colors, and
# mark + label each center of mass. Plotly so you can zoom/pan. Colors match the FFT plot
# (SAMPLE_COLORS). com is stored as (row, col); the empty-box sentinel com == [-1, -1] is
# skipped for the COM marker.
fig = go.Figure()
for sample in df.itertuples():
    color = SAMPLE_COLORS[SAMPLE_COLOR_IDX[sample.sample_id]]
    mask = np.array(Image.open(sample.mask_path).convert('L'))  # (H, W) uint8
    name = f'{sample.sample_id} (x={sample.com[0]:.1f}, y={sample.com[1]:.1f})'
    contours, _ = cv2.findContours((mask > 127).astype(np.uint8),
                                   cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    shown_legend = False
    for cnt in contours:
        pts = cnt[:, 0, :]  # (P, 2) as (x=col, y=row)
        xs = np.append(pts[:, 0], pts[0, 0])
        ys = np.append(pts[:, 1], pts[0, 1])
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', name=name,
                                 line=dict(color=color, width=1.5),
                                 legendgroup=name, showlegend=not shown_legend))
        shown_legend = True
    row, col = sample.com
    if row >= 0 and col >= 0:  # skip the empty-box sentinel
        fig.add_trace(go.Scatter(x=[col], y=[row], mode='markers+text',
                                 marker=dict(color=color, size=10, line=dict(color='black', width=1)),
                                 text=[sample.sample_id], textposition='top right',
                                 textfont=dict(color=color),
                                 legendgroup=name, showlegend=False))

fig.update_layout(title=f'Segmentation masks & COM ({BOX} box, speaker {SPEAKER})',
                  xaxis_title='col (x)', yaxis_title='row (y)',
                  legend_title='sample_id (com)', width=800, height=400)
fig.update_yaxes(autorange='reversed', scaleanchor='x', scaleratio=1)  # image coords + square pixels
fig.show()

In [48]:
import numpy as np
import plotly.graph_objects as go
# Plot the FFT magnitude spectrum for selected samples (one line per sample) in Plotly so
# you can zoom/pan. 03_fft_shifts.npz stores the complex 'fft' (1, L, F, 2) and the matching
# 'freqs' (F,) in Hz, so the frequency axis is already correct for the camera sampling rate.
#
# sample_ids selects which samples to plot (default None -> all samples in df); ints like
#   [1, 41] are accepted too and get zero-padded to ['000001', '000041'].
# laser_idx / xy_idx default to None, meaning average over that axis:
#   laser_idx=None -> average over all 100 lasers; laser_idx=5 -> only laser 5.
#   xy_idx=None    -> average over the x and y directions; xy_idx=0 -> only x.
def plot_fft_magnitude(sample_ids=None, laser_idx=None, xy_idx=None):
    if sample_ids is None:
        sub = df
    else:
        width = df['sample_id'].str.len().max()
        wanted = [str(s).zfill(width) for s in sample_ids]
        sub = df[df['sample_id'].isin(wanted)]
    fig = go.Figure()
    for sample in sub.itertuples():
        with np.load(sample.fft_path) as d:
            fft = d['fft']      # (1, L, F, 2) complex
            freqs = d['freqs']  # (F,) Hz
        mag = np.abs(fft[0])    # (L, F, 2)
        mag = mag.mean(axis=0) if laser_idx is None else mag[laser_idx]  # (F, 2)
        mag = mag.mean(axis=1) if xy_idx is None else mag[:, xy_idx]     # (F,)
        name = f'{sample.sample_id} (x={sample.com[0]:.1f}, y={sample.com[1]:.1f})'
        fig.add_trace(go.Scatter(x=freqs, y=mag, mode='lines', name=name,
                                 line=dict(color=SAMPLE_COLORS[SAMPLE_COLOR_IDX[sample.sample_id]], width=1)))
    laser_str = 'avg' if laser_idx is None else laser_idx
    xy_str = 'avg' if xy_idx is None else xy_idx
    fig.update_layout(
        title=f'FFT magnitude ({BOX} box, speaker {SPEAKER}) | laser={laser_str}, xy={xy_str}',
        xaxis_title='frequency (Hz)', yaxis_title='FFT magnitude',
        legend_title='sample_id (com)', width=1200, height=600)
    fig.show()


plot_fft_magnitude()

In [49]:
def normalized_cross_correlation(fft_a, fft_b):
    """NCC of two complex FFTs: |<a, b>| / (||a|| * ||b||).

    The inner product is the complex (Hermitian) inner product sum(conj(a) * b),
    normalized by the product of the L2 norms of each FFT. We return the
    magnitude, giving a similarity in [0, 1]. Computed in complex128 so a
    sequence's NCC with itself is exactly 1.0 (complex64 rounds slightly above).
    """
    a = np.asarray(fft_a, dtype=np.complex128).ravel()
    b = np.asarray(fft_b, dtype=np.complex128).ravel()
    inner = np.vdot(a, b)  # = sum(conj(a) * b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return np.abs(inner) / denom


def load_fft(path):
    """Load the complex FFT array from a 03_fft_shifts.npz file."""
    with np.load(path) as d:
        return d['fft']

In [50]:
# Precompute the FFT for each sample once (loading is the expensive part).
ffts = {row.sample_id: load_fft(row.fft_path) for row in df.itertuples()}
coms = {row.sample_id: row.com for row in df.itertuples()}

pair_rows = []
for sid1, sid2 in combinations(df['sample_id'], 2):
    ncc = normalized_cross_correlation(ffts[sid1], ffts[sid2])
    # Empty-box pair (com == [-1, -1]) has no real position: treat it as very far.
    if np.any(coms[sid1] == -1) or np.any(coms[sid2] == -1):
        com_dist = 200.0
    else:
        com_dist = np.linalg.norm(coms[sid1] - coms[sid2])
    pair_rows.append({
        'sample_id1': sid1,
        'sample_id2': sid2,
        'com1': coms[sid1],
        'com2': coms[sid2],
        'com_dist': com_dist,
        'ncc': ncc,
    })

pair_df = pd.DataFrame(pair_rows)
pair_df

,sample_id1,sample_id2,com1,com2,com_dist,ncc
0,000073,000065,"[-1.0, -1.0]","[72.5762987012987, 134.17694805194805]",200.000000,0.904059
1,000073,000017,"[-1.0, -1.0]","[73.36605657237936, 182.27787021630616]",200.000000,0.985030
2,000073,000041,"[-1.0, -1.0]","[74.52380952380952, 157.4055829228243]",200.000000,0.768919
3,000073,000057,"[-1.0, -1.0]","[93.28410914927768, 132.93258426966293]",200.000000,0.956442
4,000073,000009,"[-1.0, -1.0]","[94.10726072607261, 181.04620462046205]",200.000000,0.946070
5,000073,000033,"[-1.0, -1.0]","[94.29256198347107, 156.5586776859504]",200.000000,0.955726
6,000073,000001,"[-1.0, -1.0]","[113.23205342237061, 178.35559265442404]",200.000000,0.979130
7,000073,000025,"[-1.0, -1.0]","[113.29883138564274, 154.2220367278798]",200.000000,0.950809
8,000073,000049,"[-1.0, -1.0]","[114.24592833876221, 131.77524429967426]",200.000000,0.844021
9,000065,000017,"[72.5762987012987, 134.17694805194805]","[73.36605657237936, 182.27787021630616]",48.107405,0.953778


In [51]:
# Sum of NCC over the off-diagonal cells shown in the heatmap (lower triangle).
# These are exactly the unique pairs, which is what pair_df holds.
off_diag_ncc_sum = pair_df['ncc'].sum()
off_diag_ncc_sum

42.68661005588919

In [52]:
import numpy as np
import plotly.graph_objects as go

# Build a symmetric NCC matrix indexed by sample_id (diagonal = self-NCC = 1).
sample_ids = list(df['sample_id'])
n = len(sample_ids)
ncc_mat = pd.DataFrame(np.eye(n), index=sample_ids, columns=sample_ids)
for r in pair_df.itertuples():
    ncc_mat.loc[r.sample_id1, r.sample_id2] = r.ncc
    ncc_mat.loc[r.sample_id2, r.sample_id1] = r.ncc

# Symmetric, so blank out the upper triangle (above the diagonal) to drop duplicate pairs.
z = ncc_mat.values.astype(float)
z[np.triu(np.ones((n, n), dtype=bool), k=1)] = np.nan
text = [['' if np.isnan(z[i, j]) else f'{z[i, j]:.2f}' for j in range(n)] for i in range(n)]

# Tick labels: sample_id then COM (rounded to nearest tenth) on separate lines.
tick_labels = [f'{sid}<br>x={coms[sid][0]:.1f}<br>y={coms[sid][1]:.1f}' for sid in sample_ids]
off_diag_ncc_sum = pair_df['ncc'].sum()

fig = go.Figure(go.Heatmap(
    z=z, x=list(range(n)), y=list(range(n)), zmin=0, zmax=1, colorscale='Blues',
    text=text, texttemplate='%{text}', textfont=dict(size=9),
    colorbar=dict(title='NCC'), hoverongaps=False))
fig.update_layout(
    title=f'NCC ({BOX} box, speaker {SPEAKER}) | off-diagonal sum = {off_diag_ncc_sum:.2f}',
    xaxis_title='sample_id', yaxis_title='sample_id', width=900, height=800)
fig.update_xaxes(tickvals=list(range(n)), ticktext=tick_labels, tickangle=90)
fig.update_yaxes(tickvals=list(range(n)), ticktext=tick_labels, autorange='reversed',
                 scaleanchor='x', scaleratio=1)
fig.show()

In [53]:
import numpy as np
import plotly.graph_objects as go

# Same layout as the NCC heatmap, colored by the COM distance between pairs. Keep all
# sample_ids on the axes (so it lines up with the NCC heatmap), but blank out the cells
# involving a com == [-1, -1] sample (the empty-box edge case).
invalid = np.array([np.all(coms[sid] == -1) for sid in sample_ids])
dist_mat = pd.DataFrame(np.zeros((n, n)), index=sample_ids, columns=sample_ids)
for r in pair_df.itertuples():
    dist_mat.loc[r.sample_id1, r.sample_id2] = r.com_dist
    dist_mat.loc[r.sample_id2, r.sample_id1] = r.com_dist

# Mask the upper triangle (duplicates) and any row/col of an invalid sample.
mask = np.triu(np.ones((n, n), dtype=bool), k=1) | invalid[:, None] | invalid[None, :]
z = dist_mat.values.astype(float)
z[mask] = np.nan
vmax = np.nanmax(z)
cum_dist = np.nansum(z)  # cumulative COM distance over the shown (valid, lower-triangle) pairs
text = [['' if np.isnan(z[i, j]) else f'{z[i, j]:.0f}' for j in range(n)] for i in range(n)]

fig = go.Figure(go.Heatmap(
    z=z, x=list(range(n)), y=list(range(n)), zmin=0, zmax=vmax, colorscale='Blues',
    text=text, texttemplate='%{text}', textfont=dict(size=9),
    colorbar=dict(title='COM distance'), hoverongaps=False))
fig.update_layout(
    title=f'COM distance ({BOX} box, speaker {SPEAKER}) | cumulative = {cum_dist:.1f}',
    xaxis_title='sample_id', yaxis_title='sample_id', width=900, height=800)
fig.update_xaxes(tickvals=list(range(n)), ticktext=tick_labels, tickangle=90)
fig.update_yaxes(tickvals=list(range(n)), ticktext=tick_labels, autorange='reversed',
                 scaleanchor='x', scaleratio=1)
fig.show()

In [54]:
import plotly.graph_objects as go

# Scatter of NCC vs COM distance for every position pair. Hover shows both sample ids and
# each sample's (x, y) center of mass.
hover = [
    f'{r.sample_id1} (x={r.com1[0]:.1f}, y={r.com1[1]:.1f})<br>'
    f'{r.sample_id2} (x={r.com2[0]:.1f}, y={r.com2[1]:.1f})<br>'
    f'com_dist={r.com_dist:.1f}, ncc={r.ncc:.3f}'
    for r in pair_df.itertuples()
]

fig = go.Figure(go.Scatter(
    x=pair_df['com_dist'], y=pair_df['ncc'], mode='markers',
    marker=dict(size=8, opacity=0.8),
    text=hover, hoverinfo='text'))
fig.update_layout(
    title=f'NCC vs COM distance ({BOX} box, speaker {SPEAKER})',
    xaxis_title='COM distance', yaxis_title='NCC', width=800, height=500)
fig.show()